# 01 — Exploration et qualité des données

L’assistant s’appuie sur une base documentaire et sur des questions annotées pour
retrouver les informations utiles aux conseillers clientèle. Ce notebook examine
ces deux sources de données avant de construire le moteur de recherche.

**Objectif :** comprendre le contenu disponible, vérifier sa cohérence et identifier
les limites du jeu d’évaluation.

Le parcours couvre le chargement et la validation des données, les paramètres des
expériences, la lecture des documents, puis l’analyse des questions et de leur
couverture documentaire. Il permet de distinguer les questions avec source de celles
pour lesquelles l’assistant devra s’abstenir.


## 1. Charger et valider les données

La validation vérifie la structure des documents et des annotations, l’unicité des
identifiants et l’existence des sources attendues. Les textes sont conservés dans
leur forme originale pour permettre leur inspection et leur citation.


In [1]:
from contextiq.ingestion.loader import DataLoader

# Détection de data/raw depuis la racine du projet ou le dossier notebooks.
# Pour des données ailleurs : DataLoader(data_dir=...) ou CONTEXTIQ_DATA_DIR.
loader = DataLoader()
dataset = loader.load()

## 2. Identifier les paramètres des expériences

Les paramètres affichés donnent les repères des analyses suivantes : le nombre de
documents recherchés, le coefficient de saturation de BM25 et les valeurs de K
retenues pour comparer les classements.


In [2]:
from contextiq.settings import Settings

settings = Settings.load()

print("CONFIGURATION CHARGÉE")
print("-" * 50)
print("Top K de recherche :", settings.retrieval.top_k)
print("Paramètre BM25 k1  :", settings.retrieval.bm25.k1)
print("K évalués          :", settings.evaluation.k_values)

CONFIGURATION CHARGÉE
--------------------------------------------------
Top K de recherche : 3
Paramètre BM25 k1  : 1.5
K évalués          : [1, 2, 3]


## 3. Mesurer le volume des données et la répartition des questions

Le nombre de documents décrit la taille du corpus. La répartition des questions
indique combien disposent d’une source de référence et combien sont hors périmètre.
Une liste `gold_docs` vide correspond à une question sans réponse dans la base ;
ces exemples sont nécessaires pour évaluer les décisions de refus.


In [3]:
documents = dataset.documents
questions = dataset.questions

in_scope = [question for question in questions if question.gold_docs]
out_of_scope = [question for question in questions if not question.gold_docs]

print(f"Documents : {len(documents)}")
print(f"Questions : {len(questions)}")
print(f"Questions avec source : {len(in_scope)}")
print(f"Questions hors périmètre : {len(out_of_scope)}")

Documents : 6
Questions : 6
Questions avec source : 5
Questions hors périmètre : 1


## 4. Examiner le contenu et la longueur des documents

La lecture des sources permet de repérer les thèmes couverts et les conditions
commerciales à préserver : montants, plafonds, délais et critères d’éligibilité.
Le nombre de mots fournit un premier repère sur la longueur des documents et sur
l’intérêt éventuel d’un découpage en passages.


In [4]:
for document in documents:
    print(f"{document.id} — {len(document.text.split())} mots")
    print(document.text)
    print()

D1 — 20 mots
Un client peut résilier son contrat mensuel à tout moment avec un préavis de 10 jours. Aucune pénalité n'est appliquée.

D2 — 22 mots
La résiliation d'un contrat annuel avant son terme entraîne des frais égaux à 50 % des mensualités restantes, plafonnés à 150 euros.

D3 — 21 mots
En cas de déménagement, le client peut résilier sans frais son contrat annuel ou biannuel sur présentation d'un justificatif de domicile.

D4 — 28 mots
Les clients en retard de paiement de plus de 30 jours voient leur ligne suspendue. La réactivation nécessite le règlement de la dette et 15 euros de frais.

D5 — 27 mots
Une offre de rétention de 3 mois à -30 % peut être proposée uniquement aux clients ayant plus de 12 mois d'ancienneté et sans retard de paiement.

D6 — 18 mots
Les données personnelles des clients ne doivent jamais être communiquées par téléphone sans vérification d'identité en deux étapes.



## 5. Examiner les questions et leurs sources de référence

Chaque question est associée aux identifiants des documents attendus. Ces annotations
permettent d’évaluer la recherche, mais ne sont pas utilisées pour construire l’index.
Leur lecture révèle aussi les différences de formulation entre les questions et les sources.


In [5]:
for question in questions:
    print(question.q)
    print("Sources attendues :", question.gold_docs or "Hors périmètre")
    print()

Combien de temps de préavis pour un contrat mensuel ?
Sources attendues : ['D1']

Quels frais si je quitte mon abonnement annuel avant la fin ?
Sources attendues : ['D2']

Je déménage, puis-je partir sans payer ?
Sources attendues : ['D3']

Ma ligne est coupée, que faire ?
Sources attendues : ['D4']

Puis-je proposer une remise à un client qui veut partir ?
Sources attendues : ['D5']

Quel est le cours de l'action de l'entreprise ?
Sources attendues : Hors périmètre



## 6. Vérifier la couverture documentaire de l’évaluation

La couverture indique quels documents sont représentés dans les annotations.
Un document sans question dédiée ne sera pas directement évalué. La quantité et la
diversité des exemples hors périmètre conditionnent également la portée de
l’évaluation de l’abstention.


In [6]:
covered_ids = {source for question in questions for source in question.gold_docs}
uncovered_ids = sorted(document.id for document in documents if document.id not in covered_ids)
print("Documents sans annotation de retrieval :", uncovered_ids)

Documents sans annotation de retrieval : ['D6']


## Lecture des résultats

Cette exploration permet de préciser les informations auxquelles l’assistant peut
répondre et celles pour lesquelles les annotations sont encore insuffisantes.
Une base valide ne garantit pas un jeu d’évaluation représentatif : la couverture
des thèmes et des formulations doit être examinée avec les volumes observés.

Le [notebook de recherche documentaire](02_retrieval_experiments.ipynb) étudie ensuite
comment retrouver ces sources à partir d’une question.
